# Imports

In [ ]:
# widen jupyter notebook window
from IPython.display import display, HTML
display(HTML("<style>.container {width:95% !important; }</style>"))

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

In [3]:
import roicat

# Find paths to data

##### Prepare list of paths to data

In this example we are using suite2p output files, but other data types can be used (CaImAn, etc.) \
See the notebook on ingesting diverse data: https://github.com/RichieHakim/ROICaT/blob/main/notebooks/jupyter/other/demo_custom_data_importing.ipynb

Make a list containing the paths to all the input files.

In this example we are using suite2p, so the following are defined:
1. `paths_allStat`: a list to all the stat.npy files
2. `paths_allOps`: a list with ops.npy files that correspond 1-to-1 with the stat.npy files

In [ ]:
dir_allOuterFolders = r'/media/rich/bigSSD/analysis_data/face_rhythm/mouse_0322R/stat_and_ops/'

pathSuffixToStat = 'stat.npy'
pathSuffixToOps = 'ops.npy'

paths_allStat = roicat.helpers.find_paths(
    dir_outer=dir_allOuterFolders,
    reMatch=pathSuffixToStat,
    depth=4,
)[:]
paths_allOps  = np.array([Path(path).resolve().parent / pathSuffixToOps for path in paths_allStat])[:]

print(f'paths to all stat files:');
[print(path) for path in paths_allStat];
print('');
print(f'paths to all ops files:');
[print(path) for path in paths_allOps];


**Important parameters**:

- `um_per_pixel` (float):
    - Resolution. 'micrometers per pixel' of the imaging field of view.

In [ ]:
data = roicat.data_importing.Data_suite2p(
    paths_statFiles=paths_allStat,
    paths_opsFiles=paths_allOps,
    um_per_pixel=1.5,  
    new_or_old_suite2p='new',
    type_meanImg='meanImgE',
    verbose=True,
)

assert data.check_completeness(verbose=False)['classification_inference'], f"Data object is missing attributes necessary for tracking."

# Classify

Everything needed for inference — the ROInet embedder, its weights, the preprocessing
configuration, the logistic regression, and the label names — is in the single
`.roicat_classifier` packet saved at the end of the
[B2 training notebook](https://github.com/RichieHakim/ROICaT/blob/main/notebooks/classification/B2_classifier_train_interactive.ipynb).

Loading it and calling `predict` reproduces the training-time pipeline exactly. There is
deliberately nothing here to re-specify: getting the network URL, the `forward_pass_version`, or
the preprocessing constants wrong would produce plausible-looking but wrong labels with no error,
so those are recorded in the packet rather than typed again.

The one thing you *do* pass is `um_per_pixel`, because it is a property of *your* data. It does not
have to match the training data's resolution — the scale normalization exists precisely to cancel
that difference.

##### 1. Load the packet

In [ ]:
DEVICE = roicat.helpers.set_device(use_GPU=True, verbose=True)

packet = roicat.classification.ClassifierPackage.load(
    path=r'/media/rich/bigSSD/data_tmp/test_data/mouse_1.roicat_classifier',
    device=DEVICE,  ## Which torch device to run the embedder on ('cpu', 'cuda', etc.)
)

print(f"label_names:           {packet.label_names}")
print(f"size_images_in:        {packet.preprocessing['size_images_in']}  <- your raw ROI images must be this size")
print(f"um_per_pixel_training: {packet.preprocessing['um_per_pixel_training']}  <- provenance only; NOT used at predict time")
print(f"latent_dim:            {packet.latent_dim}")

## Which ROInet is inside. Worth reading: a classifier is only valid with the network it was
## trained on, and the number of features does not identify a network (the tracking release
## gives 128 features at forward_pass_version='latent', same as the classification release).
print(f"\nnetwork in this packet:")
for key, value in packet.embedder_identity.items():
    if key != 'release_note':
        print(f"  {key:22s} {value}")

## Your raw ROI images must be the same height/width the packet was trained on, because
## the scale normalization is defined relative to that size. Check it here rather than
## discovering it inside predict().
ROI_images_cat = np.concatenate(data.ROI_images, axis=0)
size_expected = tuple(packet.preprocessing['size_images_in'])
print(f"\nyour ROI images:       {ROI_images_cat.shape}")
assert ROI_images_cat.shape[1:] == size_expected, (
    f"Your ROI images are {ROI_images_cat.shape[1:]} but this packet was trained on {size_expected}. "
    f"Re-import with matching image sizes, e.g. "
    f"data.transform_spatialFootprints_to_ROIImages(out_height_width={size_expected})."
)

## um_per_pixel is a property of the data, and predict() takes a single value, so every
## session passed in one call must share a resolution. See the note below if yours don't.
um_per_pixel_unique = np.unique(np.asarray(data.um_per_pixel, dtype=np.float64))
assert len(um_per_pixel_unique) == 1, (
    f"Your sessions have different um_per_pixel values ({um_per_pixel_unique.tolist()}), so they "
    f"cannot be classified in one predict() call. Predict per session instead - see the note below."
)
um_per_pixel = float(um_per_pixel_unique[0])
print(f"um_per_pixel of data:  {um_per_pixel}")

##### 2. Check ROI image sizes

In general, you want to see that a neuron fills roughly 25-50% of the area of the image. \
**Adjust `um_per_pixel` in the data importing cell above to rescale image size**

These are the images after the packet's scale normalization, i.e. exactly what the embedder will
see (before the final resize to the network's input size). The same `um_per_pixel` is used here as
in the `predict` call below, so what you inspect is what gets classified.

In [ ]:
ROI_images_rs = packet.preprocessor.scale_normalize_images(
    ROI_images=ROI_images_cat,  ## Raw ROI images, all sessions concatenated
    um_per_pixel=um_per_pixel,  ## Same value that predict() will use below
)

roicat.visualization.display_toggle_image_stack(ROI_images_rs[:1000], image_size=(200,200))

##### 3. Predict

One call: preprocess, embed, classify. Expect for large datasets (~40,000 ROIs) that this takes
around 15 minutes on CPU or 1 minute on GPU.

**If your sessions have different `um_per_pixel` values**, the assertion in the load cell above
will fail — `predict` takes a single resolution. Call it once per session and concatenate:

```python
out = [
    packet.predict(roi_images=ims, um_per_pixel=upp)
    for ims, upp in zip(data.ROI_images, data.um_per_pixel)
]
label_ids = np.concatenate([o[0] for o in out], axis=0)
probabilities = np.concatenate([o[1] for o in out], axis=0)
```

In [ ]:
label_ids, probabilities = packet.predict(
    roi_images=ROI_images_cat,  ## RAW ROI images, shape (n_rois, height, width)
    um_per_pixel=um_per_pixel,  ## Resolution of THESE images
)

## label_ids index into packet.label_names
predictions = np.array([packet.label_names[i] for i in label_ids])

results = {
    'preds': roicat.util.JSON_List([str(p) for p in predictions]),
    'label_ids': label_ids,
    'probs': probabilities,
    'label_names': roicat.util.JSON_List(packet.label_names),
}

run_data = {
    'data': data.__dict__,
    ## The packet's own record of the pipeline that produced these predictions
    'preprocessing': roicat.util.JSON_Dict(packet.preprocessing),
    'um_per_pixel_inference': um_per_pixel,
    'results': results,
}

# Visualize results

In [ ]:
u, c = np.unique(predictions, return_counts=True)

plt.figure()
plt.bar(u, c)
plt.xlabel('predicted class')
plt.ylabel('counts')

# Save outputs

Specify save location

In [21]:
dir_save = '/media/rich/bigSSD/data_tmp/test_data'
filename_prefix = 'mouse_1'

In [ ]:
paths_save = {
    'preds':    Path(dir_save) / f'{filename_prefix}.classification_inference.preds.json',
    'results':  Path(dir_save) / f'{filename_prefix}.classification_inference.results.richfile.zip',
    'run_data': Path(dir_save) / f'{filename_prefix}.classification_inference.run_data.richfile.zip',
}

roicat.helpers.json_save(obj=results['preds'], filepath=str(paths_save['preds']))
roicat.util.RichFile_ROICaT(path=paths_save['results'], backend='zip').save(results, overwrite=True)
roicat.util.RichFile_ROICaT(path=paths_save['run_data'], backend='zip').save(run_data, overwrite=True)